In [13]:
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

FEATURES = [
    "months_active", "bill_payment_ratio", "qr_transaction_consistency",
    "airtime_topup_frequency", "psychometric_score", "network_trust_score", "transaction_volatility",
    "days_since_last_transaction", "community_fraud_flag"
]

df=pd.read_csv("synthetic_merchants.csv")
# 1. Split data into inputs (X) and targets (y)
X = df[FEATURES]
y = df["risk_band"]

# 2. Split into 80% Training data and 20% Testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Initialize the XGBoost Model
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    eval_metric="mlogloss",
    early_stopping_rounds=20,
    random_state=42
)

# 4. Train the model
print("⏳ Training XGBoost model...")
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# 5. Calculate and print scores
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\n==========================================")
print(f"🎯 OVERALL ACCURACY: {accuracy * 100:.2f}%")
print("==========================================\n")

print("📊 Detailed Classification Metrics:")
print(classification_report(
    y_test, y_pred,
    target_names=["Refused", "Silver", "Gold", "Platinum"]
))



⏳ Training XGBoost model...

🎯 OVERALL ACCURACY: 95.75%

📊 Detailed Classification Metrics:
              precision    recall  f1-score   support

     Refused       1.00      1.00      1.00        60
      Silver       0.98      0.97      0.98       120
        Gold       0.92      0.96      0.94       140
    Platinum       0.95      0.90      0.92        80

    accuracy                           0.96       400
   macro avg       0.96      0.96      0.96       400
weighted avg       0.96      0.96      0.96       400



In [14]:
import joblib
import os
import pandas as pd

# 1. Create a home for your models
os.makedirs("models", exist_ok=True)

# 2. Save the trained model binary
joblib.dump(model, "models/trustlayer_xgb.pkl")
print("💾 Success! Model saved securely at: models/trustlayer_xgb.pkl")

# 3. Print Feature Importance Ranking
importances = model.feature_importances_
feature_ranking = sorted(zip(FEATURES, importances), key=lambda x: x[1], reverse=True)

print("\n👑 FEATURE IMPORTANCE RANKING:")
print("=" * 40)
for rank, (feature, score) in enumerate(feature_ranking, 1):
    print(f"{rank}. {feature:<30} -> {score * 100:.2f}%")
print("=" * 40)


💾 Success! Model saved securely at: models/trustlayer_xgb.pkl

👑 FEATURE IMPORTANCE RANKING:
1. bill_payment_ratio             -> 22.16%
2. days_since_last_transaction    -> 15.54%
3. months_active                  -> 15.41%
4. qr_transaction_consistency     -> 14.75%
5. airtime_topup_frequency        -> 11.16%
6. psychometric_score             -> 8.67%
7. network_trust_score            -> 7.76%
8. transaction_volatility         -> 4.11%
9. community_fraud_flag           -> 0.43%
